In [2]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score

In [3]:
data = load_breast_cancer()
df = pd.DataFrame(data.data,columns = data.feature_names)
df["target"] = data.target
print(df.shape)
print(df.head())

(569, 31)
   mean radius  mean texture  ...  worst fractal dimension  target
0        17.99         10.38  ...                  0.11890       0
1        20.57         17.77  ...                  0.08902       0
2        19.69         21.25  ...                  0.08758       0
3        11.42         20.38  ...                  0.17300       0
4        20.29         14.34  ...                  0.07678       0

[5 rows x 31 columns]


In [4]:
X = df.drop("target",axis=1)
y = df["target"]

In [5]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
rf_model = RandomForestClassifier(n_estimators=100,random_state=42)
rf_model.fit(X_train,y_train)
rf_pred = rf_model.predict(X_test)
print("accuracy : ",accuracy_score(y_test,rf_pred))
print("precision : ",precision_score(y_test,rf_pred))
print("recall : ",recall_score(y_test,rf_pred))
print("f1_score : ", f1_score(y_test,rf_pred))

accuracy :  0.956140350877193
precision :  0.958904109589041
recall :  0.9722222222222222
f1_score :  0.9655172413793104


In [6]:
print("Number of trees:", len(rf_model.estimators_))
print(rf_model.estimators_[0])

Number of trees: 100
DecisionTreeClassifier(max_features='sqrt', random_state=1608637542)


In [7]:
for n in [10, 50, 100, 200, 500]:

    model = RandomForestClassifier(
        n_estimators=n,
        random_state=42
    )

    model.fit(X_train, y_train)

    pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, pred)

    print(f"Trees: {n}, Accuracy: {accuracy:.4f}")

Trees: 10, Accuracy: 0.9386
Trees: 50, Accuracy: 0.9561
Trees: 100, Accuracy: 0.9561
Trees: 200, Accuracy: 0.9561
Trees: 500, Accuracy: 0.9561


In [8]:
from sklearn.model_selection import cross_val_score
rf_cv = RandomForestClassifier(n_estimators=100,random_state=42)
cv_scores = cross_val_score(rf_cv,X_train,y_train,cv=5,scoring="accuracy")
print("scores : ",cv_scores)
print("scores mean : ",cv_scores.mean())

scores :  [0.96703297 0.98901099 0.92307692 0.93406593 0.95604396]
scores mean :  0.953846153846154


In [13]:
from sklearn.model_selection import GridSearchCV

rf = RandomForestClassifier(
    random_state=42
)

param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}
rf_grid = GridSearchCV(
    rf,
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)
rf_grid.fit(X_train, y_train)
print("Best parameters:", rf_grid.best_params_)
print("Best CV score:", rf_grid.best_score_)

Best parameters: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 200}
Best CV score: 0.9604395604395606


In [15]:
best_rf = rf_grid.best_estimator_
rf_test_pred = best_rf.predict(X_test)
print("Random Forest - Final Test Results")

print("Accuracy :", accuracy_score(y_test, rf_test_pred))
print("Precision:", precision_score(y_test, rf_test_pred))
print("Recall   :", recall_score(y_test, rf_test_pred))
print("F1 Score :", f1_score(y_test, rf_test_pred))

Random Forest - Final Test Results
Accuracy : 0.956140350877193
Precision: 0.958904109589041
Recall   : 0.9722222222222222
F1 Score : 0.9655172413793104


In [16]:
importances = best_rf.feature_importances_

feature_importance = pd.Series(
    importances,
    index=X.columns
).sort_values(ascending=False)

print(feature_importance.head(10))

worst perimeter         0.133100
worst area              0.128052
worst concave points    0.108107
mean concave points     0.094414
worst radius            0.090639
mean radius             0.058662
mean perimeter          0.055242
mean area               0.049938
mean concavity          0.046207
worst concavity         0.035357
dtype: float64
